# MILP as a Variable–Constraint Bipartite Graph

This notebook converts small **0–1 MILP / binary packing** instances into variable–constraint bipartite graphs and trains a coefficient-aware GNN with PyTorch Geometric-style data structures.

The goal is not to write a neural MILP solver. It illustrates two realistic uses:

1. **Warm start / primal heuristic:** the GNN predicts `P(x_i=1)` and the candidate assignment is repaired before it is passed to a solver.
2. **Branching features/scores:** LP fractionality and GNN uncertainty can be combined to rank candidate variables.

The optimization model is

\[
\max c^\top x
\]

subject to

\[
Ax\le b,\qquad x_i\in\{0,1\}.
\]

The bipartite representation is:

- `variable` nodes = decision variables,
- `constraint` nodes = linear constraints,
- `variable -> constraint` edge when \(A_{ji}\ne 0\),
- edge feature = coefficient \(A_{ji}\).

> The training instances are intentionally small. Industrial learned-MILP systems should extract solver-state features such as reduced costs, dual values, pseudo-costs, incumbent information, and local bounds from SCIP/Gurobi/another solver.


In [ ]:
import itertools
import random
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

from scipy.optimize import linprog
from torch_geometric.data import HeteroData
from torch_geometric.loader import DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


## 1. Generate small binary packing instances

The instances are small enough that we can compute an exact binary optimum by brute force and use it as a training label. We also solve the LP relaxation with `scipy.optimize.linprog` to obtain fractionality information.


In [ ]:
def solve_binary_bruteforce(c, A, b):
    n = len(c)
    best_x = None
    best_obj = -np.inf

    for bits in itertools.product([0, 1], repeat=n):
        x = np.asarray(bits, dtype=float)
        if np.all(A @ x <= b + 1e-9):
            obj = float(c @ x)
            if obj > best_obj:
                best_obj = obj
                best_x = x.copy()

    return best_x, best_obj


def solve_lp_relaxation(c, A, b):
    res = linprog(
        -c,
        A_ub=A,
        b_ub=b,
        bounds=[(0.0, 1.0)] * len(c),
        method="highs",
    )
    if not res.success:
        raise RuntimeError(res.message)
    return res.x, float(c @ res.x)


def generate_instance(n_vars=8, n_cons=3, rng=None):
    rng = np.random.default_rng() if rng is None else rng

    c = rng.integers(1, 11, size=n_vars).astype(float)
    A = rng.integers(0, 6, size=(n_cons, n_vars)).astype(float)

    # Ensure every variable participates in at least one constraint.
    for j in range(n_vars):
        if np.all(A[:, j] == 0):
            A[rng.integers(0, n_cons), j] = rng.integers(1, 6)

    # Keep constraints meaningful but feasible.
    row_sum = A.sum(axis=1)
    b = np.maximum(
        1.0,
        np.floor(row_sum * rng.uniform(0.35, 0.60, size=n_cons)),
    )

    x_opt, obj_opt = solve_binary_bruteforce(c, A, b)
    x_lp, obj_lp = solve_lp_relaxation(c, A, b)

    return {
        "c": c,
        "A": A,
        "b": b,
        "x_opt": x_opt,
        "obj_opt": obj_opt,
        "x_lp": x_lp,
        "obj_lp": obj_lp,
    }


example = generate_instance(rng=np.random.default_rng(SEED))
example


## 2. Convert MILP data to `HeteroData`

Variable features:

- normalized objective coefficient \(c_i\),
- LP relaxation value \(x_i^{LP}\),
- LP fractionality,
- variable degree.

Constraint features:

- normalized right-hand side,
- LP slack,
- constraint degree.

Edge feature:

- normalized coefficient \(A_{ji}\).


In [ ]:
def safe_scale(x):
    x = np.asarray(x, dtype=float)
    scale = np.max(np.abs(x))
    return x / scale if scale > 0 else x


def milp_to_heterodata(inst):
    c, A, b = inst["c"], inst["A"], inst["b"]
    x_lp, x_opt = inst["x_lp"], inst["x_opt"]

    n_cons, n_vars = A.shape

    var_degree = (A != 0).sum(axis=0).astype(float) / max(1, n_cons)
    fractionality = 1.0 - 2.0 * np.abs(x_lp - 0.5)
    fractionality = np.clip(fractionality, 0.0, 1.0)

    var_x = np.column_stack([
        safe_scale(c),
        x_lp,
        fractionality,
        var_degree,
    ])

    lp_slack = b - A @ x_lp
    con_degree = (A != 0).sum(axis=1).astype(float) / max(1, n_vars)
    con_x = np.column_stack([
        b / np.maximum(A.sum(axis=1), 1.0),
        lp_slack / np.maximum(b, 1.0),
        con_degree,
    ])

    rows, cols = np.nonzero(A)
    edge_index = np.vstack([cols, rows])  # variable -> constraint
    coeff = A[rows, cols]
    coeff = coeff / max(np.max(np.abs(A)), 1.0)

    data = HeteroData()
    data["variable"].x = torch.tensor(var_x, dtype=torch.float32)
    data["variable"].y = torch.tensor(x_opt, dtype=torch.float32)
    data["constraint"].x = torch.tensor(con_x, dtype=torch.float32)

    rel = data["variable", "participates", "constraint"]
    rel.edge_index = torch.tensor(edge_index, dtype=torch.long)
    rel.edge_attr = torch.tensor(coeff[:, None], dtype=torch.float32)

    return data


graph = milp_to_heterodata(example)
graph


## 3. Coefficient-aware bipartite message passing

Instead of treating the coefficient matrix only as connectivity, the message function explicitly receives the normalized `A_ji` edge coefficient.

The block performs two passes:

```text
variable --(A_ji)--> constraint
constraint --(A_ji)--> variable
```


In [ ]:
def mean_aggregate(messages, index, dim_size):
    out = messages.new_zeros((dim_size, messages.size(-1)))
    out.index_add_(0, index, messages)

    count = messages.new_zeros((dim_size, 1))
    ones = messages.new_ones((messages.size(0), 1))
    count.index_add_(0, index, ones)

    return out / count.clamp_min(1.0)


class BipartiteBlock(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()

        self.var_to_con = nn.Sequential(
            nn.Linear(hidden_dim + 1, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.update_con = nn.Sequential(
            nn.Linear(2 * hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        self.con_to_var = nn.Sequential(
            nn.Linear(hidden_dim + 1, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.update_var = nn.Sequential(
            nn.Linear(2 * hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        self.var_norm = nn.LayerNorm(hidden_dim)
        self.con_norm = nn.LayerNorm(hidden_dim)

    def forward(self, h_var, h_con, edge_index, edge_attr):
        var_idx, con_idx = edge_index

        msg_vc = self.var_to_con(
            torch.cat([h_var[var_idx], edge_attr], dim=-1)
        )
        agg_con = mean_aggregate(msg_vc, con_idx, h_con.size(0))
        new_con = self.con_norm(
            h_con + self.update_con(torch.cat([h_con, agg_con], dim=-1))
        )

        msg_cv = self.con_to_var(
            torch.cat([new_con[con_idx], edge_attr], dim=-1)
        )
        agg_var = mean_aggregate(msg_cv, var_idx, h_var.size(0))
        new_var = self.var_norm(
            h_var + self.update_var(torch.cat([h_var, agg_var], dim=-1))
        )

        return new_var, new_con


class BipartiteMILPGNN(nn.Module):
    def __init__(self, hidden_dim=64, layers=3):
        super().__init__()
        self.var_encoder = nn.Linear(4, hidden_dim)
        self.con_encoder = nn.Linear(3, hidden_dim)
        self.blocks = nn.ModuleList(
            [BipartiteBlock(hidden_dim) for _ in range(layers)]
        )
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, data):
        h_var = torch.relu(self.var_encoder(data["variable"].x))
        h_con = torch.relu(self.con_encoder(data["constraint"].x))

        rel = data["variable", "participates", "constraint"]
        for block in self.blocks:
            h_var, h_con = block(
                h_var,
                h_con,
                rel.edge_index,
                rel.edge_attr,
            )

        return self.head(h_var).squeeze(-1)


## 4. Build a small supervised dataset

Exact binary optima are used only because these educational instances are tiny. In serious learned-branching work, labels are commonly generated from solver policies such as strong branching rather than from brute-force optimal variable assignments.


In [ ]:
rng = np.random.default_rng(SEED)

train_instances = [
    generate_instance(n_vars=8, n_cons=3, rng=rng)
    for _ in range(160)
]
test_instances = [
    generate_instance(n_vars=8, n_cons=3, rng=rng)
    for _ in range(40)
]

train_graphs = [milp_to_heterodata(inst) for inst in train_instances]
test_graphs = [milp_to_heterodata(inst) for inst in test_instances]

train_loader = DataLoader(train_graphs, batch_size=16, shuffle=True)

model = BipartiteMILPGNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)

positives = sum(float(g["variable"].y.sum()) for g in train_graphs)
total = sum(g["variable"].y.numel() for g in train_graphs)
pos_weight = torch.tensor(
    max((total - positives) / max(positives, 1.0), 1.0),
    device=device,
)

for epoch in range(80):
    model.train()
    running = 0.0

    for batch in train_loader:
        batch = batch.to(device)
        logits = model(batch)
        y = batch["variable"].y

        loss = F.binary_cross_entropy_with_logits(
            logits,
            y,
            pos_weight=pos_weight,
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running += float(loss)

    if (epoch + 1) % 20 == 0:
        print(f"epoch={epoch+1:3d} loss={running/len(train_loader):.4f}")


## 5. Turn GNN probabilities into a feasible warm-start candidate

Thresholded predictions may violate `A x <= b`. A simple repair removes selected variables with the weakest predicted utility until all constraints are satisfied.


In [ ]:
@torch.no_grad()
def predict_prob(model, graph):
    model.eval()
    graph = graph.to(device)
    return torch.sigmoid(model(graph)).cpu().numpy()


def repair_packing(inst, prob, threshold=0.5):
    c, A, b = inst["c"], inst["A"], inst["b"]
    x = (prob >= threshold).astype(float)

    while np.any(A @ x > b + 1e-9):
        selected = np.flatnonzero(x > 0.5)
        if len(selected) == 0:
            break

        pressure = (A[:, selected] / np.maximum(b[:, None], 1.0)).sum(axis=0)
        score = prob[selected] * c[selected] / np.maximum(pressure, 1e-6)
        remove = selected[np.argmin(score)]
        x[remove] = 0.0

    return x


rows = []
for inst, graph in zip(test_instances, test_graphs):
    p = predict_prob(model, graph)
    x = repair_packing(inst, p)
    feasible = bool(np.all(inst["A"] @ x <= inst["b"] + 1e-9))
    obj = float(inst["c"] @ x)
    gap = 100.0 * (inst["obj_opt"] - obj) / max(abs(inst["obj_opt"]), 1e-9)
    rows.append((feasible, obj, gap))

print("Feasibility rate:", np.mean([r[0] for r in rows]))
print("Mean objective gap (%):", np.mean([r[2] for r in rows]))


## 6. A branching-candidate interpretation

This notebook does **not** implement a true learned branching policy. It shows how two useful signals can be combined:

- LP fractionality: variables near 0.5 are attractive branching candidates,
- GNN uncertainty: probabilities near 0.5 indicate uncertainty.

A production learned-branching system should train directly on strong-branching or another expert policy and should evaluate search-tree size and solve time.


In [ ]:
inst = test_instances[0]
graph = test_graphs[0]
p = predict_prob(model, graph)

lp_fractionality = 1.0 - 2.0 * np.abs(inst["x_lp"] - 0.5)
lp_fractionality = np.clip(lp_fractionality, 0.0, 1.0)

gnn_uncertainty = 1.0 - 2.0 * np.abs(p - 0.5)
gnn_uncertainty = np.clip(gnn_uncertainty, 0.0, 1.0)

candidate_score = 0.7 * lp_fractionality + 0.3 * gnn_uncertainty
ranking = np.argsort(candidate_score)[::-1]

print("LP values:", np.round(inst["x_lp"], 3))
print("GNN probabilities:", np.round(p, 3))
print("Candidate ranking:", ranking)
print("Combined scores:", np.round(candidate_score[ranking], 3))


## 7. Production path

A realistic extension is:

```text
SCIP / PySCIPOpt state
 -> variable-constraint bipartite graph
 -> solver-state features
 -> GNN
 -> branching / fixing / heuristic score
 -> solver
```

See `examples/03_pyscipopt_learned_branching.py` for a solver-in-the-loop branching example.
